<a href="https://colab.research.google.com/github/metal-carp2/EEG_Reaserch/blob/main/baseline_eegnet_loso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import dependencies

!pip install braindecode --quiet
!pip install mne


import os
import glob
import numpy as np
import scipy.io
import mne
import time
import copy
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from braindecode.models import EEGNet
from tqdm.auto import tqdm
from scipy.stats import wilcoxon
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.9/626.9 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 145.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 140.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
moviepy 1.0.3 re

In [2]:
#download data

!wget https://www.bbci.de/competition/download/competition_iv/BCICIV_2a_gdf.zip

--2026-07-13 18:00:42--  https://www.bbci.de/competition/download/competition_iv/BCICIV_2a_gdf.zip
Resolving www.bbci.de (www.bbci.de)... 141.23.71.83
Connecting to www.bbci.de (www.bbci.de)|141.23.71.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 439968864 (420M) [application/zip]
Saving to: ‘BCICIV_2a_gdf.zip’

BCICIV_2a_gdf.zip   100%[===================>] 419.59M   189MB/s    in 2.2s    

2026-07-13 18:00:45 (189 MB/s) - ‘BCICIV_2a_gdf.zip’ saved [439968864/439968864]



In [3]:
!mkdir -p /content/cleaned_data/
!unzip /content/BCICIV_2a_gdf.zip -d /content/raw_data


raw_files = sorted(glob.glob('/content/raw_data/**/*.gdf', recursive=True))
print(len(raw_files))
print(raw_files[:5])
!wget https://www.bbci.de/competition/iv/results/ds2a/true_labels.zip
!unzip true_labels.zip -d /content/true_labels


Archive:  /content/BCICIV_2a_gdf.zip
  inflating: /content/raw_data/A01E.gdf  
  inflating: /content/raw_data/A01T.gdf  
  inflating: /content/raw_data/A02E.gdf  
  inflating: /content/raw_data/A02T.gdf  
  inflating: /content/raw_data/A03E.gdf  
  inflating: /content/raw_data/A03T.gdf  
  inflating: /content/raw_data/A04E.gdf  
  inflating: /content/raw_data/A04T.gdf  
  inflating: /content/raw_data/A05E.gdf  
  inflating: /content/raw_data/A05T.gdf  
  inflating: /content/raw_data/A06E.gdf  
  inflating: /content/raw_data/A06T.gdf  
  inflating: /content/raw_data/A07E.gdf  
  inflating: /content/raw_data/A07T.gdf  
  inflating: /content/raw_data/A08E.gdf  
  inflating: /content/raw_data/A08T.gdf  
  inflating: /content/raw_data/A09E.gdf  
  inflating: /content/raw_data/A09T.gdf  
18
['/content/raw_data/A01E.gdf', '/content/raw_data/A01T.gdf', '/content/raw_data/A02E.gdf', '/content/raw_data/A02T.gdf', '/content/raw_data/A03E.gdf']
--2026-07-13 18:00:52--  https://www.bbci.de/competit

In [5]:
raw_data_folder = '/content/raw_data/'
true_labels_folder = '/content/true_labels/'

test_file = sorted([f for f in os.listdir(raw_data_folder) if f.endswith('E.gdf')])[0]
file_path = os.path.join(raw_data_folder, test_file)
raw = mne.io.read_raw_gdf(file_path, eog=['EOG-left', 'EOG-central', 'EOG-right'], preload=True)
events, event_id = mne.events_from_annotations(raw)
print(event_id)
print(len(events))

label_file = os.path.join(true_labels_folder, test_file[:-4] + '.mat')
label_data = scipy.io.loadmat(label_file)
print(label_data.keys())
true_labels = label_data['classlabel'].squeeze()
print(len(true_labels))

Extracting GDF parameters from /content/raw_data/A01E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Reading 0 ... 686999  =      0.000 ...  2747.996 secs...
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('783'): 7}
595
dict_keys(['__header__', '__version__', '__globals__', 'classlabel'])
288


In [6]:
cue_event_id = {k: v for k, v in event_id.items() if k == '783'}
cue_events = events[np.isin(events[:, 2], list(cue_event_id.values()))]
print(len(cue_events))
print(len(true_labels))

288
288


In [8]:
os.makedirs('/content/cleaned_data/second_session', exist_ok=True)

raw_data_folder = '/content/raw_data/'
cleaned_data_folder = '/content/cleaned_data/second_session/'
true_labels_folder = '/content/true_labels/'
filtered_files = sorted([f for f in os.listdir(raw_data_folder) if f.endswith('E.gdf')])

for file in tqdm(filtered_files, desc="Preprocessing E sessions"):
    file_path = os.path.join(raw_data_folder, file)
    raw = mne.io.read_raw_gdf(file_path, eog=['EOG-left', 'EOG-central', 'EOG-right'], preload=True)
    raw.drop_channels(['EOG-left', 'EOG-central', 'EOG-right'])
    raw.filter(l_freq=4, h_freq=40, method='iir')
    raw.notch_filter(freqs=50)

    label_file = os.path.join(true_labels_folder, file[:-4] + '.mat')
    label_data = scipy.io.loadmat(label_file)
    true_labels = label_data['classlabel'].squeeze()

    events, event_id = mne.events_from_annotations(raw)
    cue_event_id = {k: v for k, v in event_id.items() if k == '783'}
    cue_events = events[np.isin(events[:, 2], list(cue_event_id.values()))]

    assert len(cue_events) == len(true_labels), \
        f"{file}: {len(cue_events)} cue events vs {len(true_labels)} labels, mismatch"

    new_file_path = os.path.join(cleaned_data_folder, file[:-4] + '.fif')
    raw.save(new_file_path, overwrite=True)

    np.save(os.path.join(cleaned_data_folder, file[:-4] + '_labels.npy'), true_labels)

Preprocessing E sessions:   0%|          | 0/9 [00:00<?, ?it/s]

Extracting GDF parameters from /content/raw_data/A01E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 686999  =      0.000 ...  2747.996 secs...


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 40 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 4.00, 40.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 1651 samples (6.604 s)

Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_

/tmp/ipykernel_8780/2495293247.py:27: RuntimeWarning: This filename (/content/cleaned_data/second_session/A01E.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(new_file_path, overwrite=True)


...

Closing /content/cleaned_data/second_session/A08E.fif
[done]
Extracting GDF parameters from /content/raw_data/A09E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 675097  =      0.000 ...  2700.388 secs...


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 40 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 4.00, 40.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 1651 samples (6.604 s)

Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_

/tmp/ipykernel_8780/2495293247.py:27: RuntimeWarning: This filename (/content/cleaned_data/second_session/A09E.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(new_file_path, overwrite=True)


Closing /content/cleaned_data/second_session/A09E.fif
[done]


In [12]:
os.makedirs('/content/cleaned_data/first_session', exist_ok=True)

raw_data_folder = '/content/raw_data/'
cleaned_data_folder = '/content/cleaned_data/first_session/'
filtered_files = sorted([f for f in os.listdir(raw_data_folder) if f.endswith('T.gdf')])

for file in tqdm(filtered_files, desc="Preprocessing T sessions"):
    file_path = os.path.join(raw_data_folder, file)
    raw = mne.io.read_raw_gdf(file_path, eog=['EOG-left', 'EOG-central', 'EOG-right'], preload=True)
    raw.drop_channels(['EOG-left', 'EOG-central', 'EOG-right'])
    raw.filter(l_freq=4, h_freq=40, method='iir')
    raw.notch_filter(freqs=50)
    new_file_path = os.path.join(cleaned_data_folder, file[:-4] + '.fif')
    raw.save(new_file_path, overwrite=True)

Preprocessing T sessions:   0%|          | 0/9 [00:00<?, ?it/s]

Extracting GDF parameters from /content/raw_data/A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 40 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 4.00, 40.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 1651 samples (6.604 s)

Writing /content/cleaned_data/first_session/A01T.fif


/tmp/ipykernel_8780/1259912685.py:14: RuntimeWarning: This filename (/content/cleaned_data/first_session/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(new_file_path, overwrite=True)


...

Closing /content/cleaned_data/first_session/A08T.fif
[done]
Extracting GDF parameters from /content/raw_data/A09T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 673327  =      0.000 ...  2693.308 secs...


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 40 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 4.00, 40.00 Hz: -6.02, -6.02 dB

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 1651 samples (6.604 s)

Writing /content/cleaned_data/first_session/A09T.fif


/tmp/ipykernel_8780/1259912685.py:14: RuntimeWarning: This filename (/content/cleaned_data/first_session/A09T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(new_file_path, overwrite=True)


Closing /content/cleaned_data/first_session/A09T.fif
[done]


In [14]:
cleaned_data_dir_T = '/content/cleaned_data/first_session/'
cleaned_data_dir_E = '/content/cleaned_data/second_session/'

dataset = {}

t_files = sorted(glob.glob(os.path.join(cleaned_data_dir_T, 'A0?T.fif')))

for fif_path in tqdm(t_files, desc="Epoching T sessions"):
    subject_id = int(os.path.basename(fif_path)[1:3])
    raw = mne.io.read_raw_fif(fif_path, preload=True)
    events, event_id = mne.events_from_annotations(raw)

    mi_event_id = {k: v for k, v in event_id.items() if k in ['769', '770', '771', '772']}

    epochs = mne.Epochs(raw, events, event_id=mi_event_id, tmin=2, tmax=6,
                         reject=None, baseline=None, preload=True)
    X = epochs.get_data(copy=True)
    y_raw = epochs.events[:, 2]

    label_map = {v: k for k, v in mi_event_id.items()}
    class_map = {'769': 1, '770': 2, '771': 3, '772': 4}
    y = np.array([class_map[label_map[code]] for code in y_raw])

    assert X.shape[0] >= 285, f"Subject {subject_id} T: expected ~288 epochs, got {X.shape[0]}"
    assert len(y) == X.shape[0], f"Subject {subject_id} T: label/epoch count mismatch"
    if X.shape[0] != 288:
        print(f"  NOTE: Subject {subject_id} T dropped {288 - X.shape[0]} epoch(s), drop_log: {[d for d in epochs.drop_log if d]}")

    dataset.setdefault(subject_id, {})['T'] = {'X': X, 'y': y}
    print(f"Subject {subject_id} T: {X.shape}, labels {np.unique(y, return_counts=True)}")


e_files = sorted(glob.glob(os.path.join(cleaned_data_dir_E, 'A0?E.fif')))

for fif_path in tqdm(e_files, desc="Epoching E sessions"):
    subject_id = int(os.path.basename(fif_path)[1:3])
    raw = mne.io.read_raw_fif(fif_path, preload=True)
    events, event_id = mne.events_from_annotations(raw)

    epochs = mne.Epochs(raw, events, event_id=event_id['783'], tmin=0, tmax=4,
                         reject=None, baseline=None, preload=True)
    X = epochs.get_data(copy=True)
    y = np.load(os.path.join(cleaned_data_dir_E, f'A{subject_id:02d}E_labels.npy'))

    assert X.shape[0] >= 285, f"Subject {subject_id} E: expected ~288 epochs, got {X.shape[0]}"
    assert len(y) == X.shape[0], f"Subject {subject_id} E: label/epoch count mismatch, {X.shape[0]} epochs vs {len(y)} labels"
    if X.shape[0] != 288:
        print(f"  NOTE: Subject {subject_id} E dropped {288 - X.shape[0]} epoch(s)")

    dataset.setdefault(subject_id, {})['E'] = {'X': X, 'y': y}
    print(f"Subject {subject_id} E: {X.shape}, labels {np.unique(y, return_counts=True)}")


import pickle
with open('/content/cleaned_data/bci_iv_2a_epoched.pkl', 'wb') as f:
    pickle.dump(dataset, f)

print("\nDataset structure: dataset[subject_id]['T' or 'E']['X'/'y']")

Epoching T sessions:   0%|          | 0/9 [00:00<?, ?it/s]

Opening raw data file /content/cleaned_data/first_session/A01T.fif...
Isotrak not found
    Range : 0 ... 672527 =      0.000 ...  2690.108 secs
Ready.
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/tmp/ipykernel_8780/3808228580.py:10: RuntimeWarning: This filename (/content/cleaned_data/first_session/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_path, preload=True)


Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1001 original time points ...
1 bad epochs dropped
  NOTE: Subject 1 T dropped 1 epoch(s), drop_log: [('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), ('IGNORED',), 

/tmp/ipykernel_8780/3808228580.py:10: RuntimeWarning: This filename (/content/cleaned_data/first_session/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_path, preload=True)


...

Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1001 original time points ...
0 bad epochs dropped
Subject 7 E: (288, 22, 1001), labels (array([1, 2, 3, 4], dtype=uint8), array([72, 72, 72, 72]))
Opening raw data file /content/cleaned_data/second_session/A08E.fif...
Isotrak not found
    Range : 0 ... 687791 =      0.000 ...  2751.164 secs
Ready.
Reading 0 ... 687791  =      0.000 ...  2751.164 secs...


/tmp/ipykernel_8780/3808228580.py:37: RuntimeWarning: This filename (/content/cleaned_data/second_session/A08E.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_path, preload=True)


Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1001 original time points ...
0 bad epochs dropped
Subject 8 E: (288, 22, 1001), labels (array([1, 2, 3, 4], dtype=uint8), array([72, 72, 72, 72]))
Opening raw data file /content/cleaned_data/second_session/A09E.fif...
Isotrak not found
    Range : 0 ... 675097 =      0.000 ...  2700.388 secs
Ready.
Reading 0 ... 675097  =      0.000 ...  2700.388 secs...


/tmp/ipykernel_8780/3808228580.py:37: RuntimeWarning: This filename (/content/cleaned_data/second_session/A09E.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_path, preload=True)


Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('783')]
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1001 original time points ...
0 bad epochs dropped
Subject 9 E: (288, 22, 1001), labels (array([1, 2, 3, 4], dtype=uint8), array([72, 72, 72, 72]))

Dataset structure: dataset[subject_id]['T' or 'E']['X'/'y']


In [15]:
print(dataset.keys())
for sid in sorted(dataset.keys()):
    print(sid, list(dataset[sid].keys()))

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9])
1 ['T', 'E']
2 ['T', 'E']
3 ['T', 'E']
4 ['T', 'E']
5 ['T', 'E']
6 ['T', 'E']
7 ['T', 'E']
8 ['T', 'E']
9 ['T', 'E']


In [17]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) - 1  # shift 1-4 to 0-3 for CrossEntropyLoss
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [24]:
def normalize_per_trial(X):
    # X shape: (n_trials, n_channels, n_times)
    mean = X.mean(axis=(1, 2), keepdims=True)
    std = X.std(axis=(1, 2), keepdims=True)
    return (X - mean) / (std + 1e-8)

train_X_norm = normalize_per_trial(train_X)
test_X_norm = normalize_per_trial(test_X)

train_ds = EEGDataset(train_X_norm, train_y)
test_ds = EEGDataset(test_X_norm, test_y)

In [25]:
val_frac = 0.15
n_val = int(len(train_ds) * val_frac)
n_train = len(train_ds) - n_val
train_sub, val_sub = random_split(train_ds, [n_train, n_val])

train_sub_loader = DataLoader(train_sub, batch_size=64, shuffle=True)
val_loader = DataLoader(val_sub, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [29]:
print(f"Epoch {epoch}: val_loss={val_loss:.4f}")
print(f"Epoch {epoch}: train_loss={loss.item():.4f}, val_loss={val_loss:.4f}")

Epoch 47: val_loss=1.4043
Epoch 47: train_loss=1.0170, val_loss=1.4043


In [18]:
ds_test = EEGDataset(dataset[1]['E']['X'], dataset[1]['E']['y'])
print(ds_test.X.shape, ds_test.y.shape)
print(ds_test.y.min().item(), ds_test.y.max().item())

torch.Size([288, 22, 1001]) torch.Size([288])
0 3


In [19]:
#LOSO fold construction

def build_loso_fold(dataset, held_out_subject):
    train_X_list, train_y_list = [], []
    for sid in dataset:
        if sid == held_out_subject:
            continue
        train_X_list.append(dataset[sid]['T']['X'])
        train_y_list.append(dataset[sid]['T']['y'])

    train_X = np.concatenate(train_X_list, axis=0)
    train_y = np.concatenate(train_y_list, axis=0)

    test_X = dataset[held_out_subject]['E']['X']
    test_y = dataset[held_out_subject]['E']['y']

    return train_X, train_y, test_X, test_y


held_out_subject = 1
train_X, train_y, test_X, test_y = build_loso_fold(dataset, held_out_subject)
print("Train:", train_X.shape, train_y.shape)
print("Test:", test_X.shape, test_y.shape)

train_ds = EEGDataset(train_X, train_y)
test_ds = EEGDataset(test_X, test_y)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

Train: (2296, 22, 1001) (2296,)
Test: (288, 22, 1001) (288,)


In [20]:
help(EEGNet)


Help on class EEGNet in module braindecode.models.eegnet:

class EEGNet(braindecode.models.base.EEGModuleMixin, torch.nn.modules.container.Sequential)
 |  EEGNet(n_chans: 'Optional[int]' = None, n_outputs: 'Optional[int]' = None, n_times: 'Optional[int]' = None, final_conv_length: 'str | int' = 'auto', pool_mode: 'str' = 'mean', F1: 'int' = 8, D: 'int' = 2, F2: 'Optional[int | None]' = None, kernel_length: 'int' = 64, *, depthwise_kernel_length: 'int' = 16, pool1_kernel_size: 'int' = 4, pool2_kernel_size: 'int' = 8, conv_spatial_max_norm: 'int' = 1, activation: 'type[nn.Module]' = <class 'torch.nn.modules.activation.ELU'>, batch_norm_momentum: 'float' = 0.01, batch_norm_affine: 'bool' = True, batch_norm_eps: 'float' = 0.001, drop_prob: 'float' = 0.25, final_layer_with_constraint: 'bool' = False, norm_rate: 'float' = 0.25, chs_info: 'Optional[list[Dict]]' = None, input_window_seconds=None, sfreq=None, **kwargs)
 |
 |  EEGNet model from Lawhern et al (2018) [Lawhern2018]_.
 |
 |  :bdg-su

In [21]:
import inspect
print(inspect.signature(EEGNet.__init__))
print(EEGNet.__doc__[:1500])

(self, n_chans: 'Optional[int]' = None, n_outputs: 'Optional[int]' = None, n_times: 'Optional[int]' = None, final_conv_length: 'str | int' = 'auto', pool_mode: 'str' = 'mean', F1: 'int' = 8, D: 'int' = 2, F2: 'Optional[int | None]' = None, kernel_length: 'int' = 64, *, depthwise_kernel_length: 'int' = 16, pool1_kernel_size: 'int' = 4, pool2_kernel_size: 'int' = 8, conv_spatial_max_norm: 'int' = 1, activation: 'type[nn.Module]' = <class 'torch.nn.modules.activation.ELU'>, batch_norm_momentum: 'float' = 0.01, batch_norm_affine: 'bool' = True, batch_norm_eps: 'float' = 0.001, drop_prob: 'float' = 0.25, final_layer_with_constraint: 'bool' = False, norm_rate: 'float' = 0.25, chs_info: 'Optional[list[Dict]]' = None, input_window_seconds=None, sfreq=None, **kwargs)
EEGNet model from Lawhern et al (2018) [Lawhern2018]_.

    :bdg-success:`Convolution`

    .. figure:: https://content.cld.iop.org/journals/1741-2552/15/5/056013/revision2/jneaace8cf01_hr.jpg
        :align: center
        :alt: E

In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

model = EEGNet(
    n_chans=22,
    n_outputs=4,
    n_times=1001,
    sfreq=250,
    kernel_length=125,
    F1=8,
    D=2
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

cuda
Total parameters: 3932


In [33]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
import time
import copy
from torch.utils.data import random_split

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

n_epochs = 500
best_val_loss = float('inf')
best_model_state = None

val_frac = 0.15
n_val = int(len(train_ds) * val_frac)
n_train = len(train_ds) - n_val
train_sub, val_sub = random_split(train_ds, [n_train, n_val])

train_sub_loader = DataLoader(train_sub, batch_size=64, shuffle=True)
val_loader = DataLoader(val_sub, batch_size=64, shuffle=False)

start_time = time.time()

for epoch in tqdm(range(n_epochs), desc=f"Training subject {held_out_subject}"):
    model.train()
    for X_batch, y_batch in train_sub_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out = model(X_batch)
            val_loss += criterion(out, y_batch).item() * X_batch.size(0)
    val_loss /= len(val_sub)

    if epoch % 20 == 0:
        print(f"Epoch {epoch}: train_loss={loss.item():.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

train_time_sec = time.time() - start_time
model.load_state_dict(best_model_state)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        out = model(X_batch)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

acc = accuracy_score(all_labels, all_preds)
kappa = cohen_kappa_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print(f"Subject {held_out_subject} | Accuracy: {acc:.4f} | Kappa: {kappa:.4f} | Train time: {train_time_sec:.1f}s")
print("Confusion matrix:\n", cm)

results = {
    'method': 'EEGNet_no_adaptation_floor',
    'held_out_subject': held_out_subject,
    'accuracy': acc,
    'kappa': kappa,
    'confusion_matrix': cm,
    'train_time_sec': train_time_sec,
    'n_epochs_trained': n_epochs,
    'total_params': total_params
}


Training subject 1:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 0: train_loss=1.2574, val_loss=1.1108
Epoch 20: train_loss=1.1131, val_loss=1.2025
Epoch 40: train_loss=0.9850, val_loss=1.2300
Epoch 60: train_loss=0.8257, val_loss=1.2500
Epoch 80: train_loss=0.8790, val_loss=1.2616
Epoch 100: train_loss=1.0019, val_loss=1.2750
Epoch 120: train_loss=0.8605, val_loss=1.3481
Epoch 140: train_loss=0.7756, val_loss=1.3594
Epoch 160: train_loss=0.9362, val_loss=1.3625
Epoch 180: train_loss=0.9748, val_loss=1.4023
Epoch 200: train_loss=0.8210, val_loss=1.4188
Epoch 220: train_loss=0.6434, val_loss=1.4175
Epoch 240: train_loss=0.7955, val_loss=1.4000
Epoch 260: train_loss=0.6442, val_loss=1.4128
Epoch 280: train_loss=0.8061, val_loss=1.4474
Epoch 300: train_loss=0.6136, val_loss=1.4107
Epoch 320: train_loss=0.6194, val_loss=1.4216
Epoch 340: train_loss=0.7849, val_loss=1.4038
Epoch 360: train_loss=0.6822, val_loss=1.4923
Epoch 380: train_loss=0.5711, val_loss=1.4804
Epoch 400: train_loss=0.5434, val_loss=1.5003
Epoch 420: train_loss=0.7679, val_loss=1